# JED Attack — Agent Security | DGX Spark GB10

**Purpose:** Run the JED five-algorithm red-team attack against the competition GGUF models
locally on an **NVIDIA DGX Spark GB10** (Grace Blackwell Superchip, 128 GB unified memory)
using llama.cpp served via Docker.

This notebook documents the full DGX workflow: Docker image build, model download,
GPT-OSS difficulties, Gemma setup, and attack execution.  It mirrors the
structure of the Kaggle evaluation notebook but runs entirely off-platform.

---

## Why run off-Kaggle?

| Reason | Detail |
|---|---|
| **Fast iteration** | No queue — start a run in seconds |
| **Budget flexibility** | Test at 300s before committing a 9000s Kaggle slot |
| **Full logs** | Complete stdout/stderr, not just the summary cell |
| **Algorithm debugging** | Read traces, fix bugs, rerun immediately |
| **GPU control** | DGX Spark GB10 has 128 GB unified memory — fits both models fully offloaded |

---

## Hardware: NVIDIA DGX Spark GB10

```
GPU:    NVIDIA GB10 Grace Blackwell Superchip
Memory: 128 GB unified (CPU + GPU)
OS:     Linux 6.17 (NVIDIA)
Docker: with --privileged + --gpus all for GPU access
```

Model VRAM requirements at Q4_0 / Q4_K_M:

| Model | Size | Fits? |
|---|---|---|
| GPT-OSS 20B Q4_K_M (unsloth) | ~9 GB | ✓ |
| Gemma 4 26B A4B Q4_0 (google) | ~14 GB | ✓ |

## 1. Docker Environment

`Dockerfile.llama` builds a CUDA-enabled container with the **llama.cpp `llama-server`
C++ binary**.  We use the server binary (not `llama-cpp-python`) because:

- `llama-cpp-python` does **not** support `--jinja`, which is required to inject the
  competition models' Jinja2 chat templates (tool schema rendering via `render_tool_namespace`).
  Without `--jinja`, models produce `!!!` / `???` garbage instead of tool calls.
- The llama.cpp C++ server handles PEG grammar-constrained sampling correctly for Gemma.

### First-time build (~20 minutes)

The first build clones the llama.cpp repo and compiles CUDA kernels from scratch:

```zsh
cd jed-redteam-attack/
docker build -t jed-llama:latest -f Dockerfile.llama .
```

Subsequent runs use the cached image and start in seconds.  Force a rebuild after
`Dockerfile.llama` changes:

```zsh
REBUILD=1 MODEL=gemma bash scripts/llama-serve.sh
```

### Image contents

The `jed-llama:latest` image contains only the `llama-server` binary and its CUDA
runtime dependencies.  No Python, no model weights — weights are mounted read-only
from the host at runtime (`-v /path/to/gguf:/models:ro`).

In [ ]:
import subprocess, json, os, sys, time, urllib.request
from pathlib import Path

REPO_DIR = Path('/home/msusol/LosusAI/Projects/Kaggle/kaggle-ai-agent-security-multi-step-tool-attacks/jed-redteam-attack')
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Check Docker image exists
result = subprocess.run(['docker', 'image', 'inspect', 'jed-llama:latest'],
                        capture_output=True)
if result.returncode == 0:
    meta = json.loads(result.stdout)[0]
    created = meta.get('Created', '')[:19]
    size_mb = round(meta.get('Size', 0) / 1e6)
    print(f'✓ jed-llama:latest exists  created={created}  size={size_mb} MB')
else:
    print('✗ jed-llama:latest NOT found')
    print('  Build with: docker build -t jed-llama:latest -f Dockerfile.llama .')

## 2. Downloading Competition Models

Models are downloaded via the HuggingFace CLI and cached on DGX's `/raid` storage.
`llama-serve.sh` handles the download automatically on first run.

### Gemma 4 26B (working ✓)

```zsh
hf download google/gemma-4-26B-A4B-it-qat-q4_0-gguf \
    gemma-4-26B_q4_0-it.gguf \
    --local-dir /raid/hf_cache/gguf/
```

File: `gemma-4-26B_q4_0-it.gguf` (~14 GB)  
Source: `google/gemma-4-26B-A4B-it-qat-q4_0-gguf` (public, no token required)

### GPT-OSS 20B (downloaded, but non-functional — see Section 3)

```zsh
hf download unsloth/gpt-oss-20b-GGUF \
    gpt-oss-20b-Q4_K_M.gguf \
    --local-dir /raid/hf_cache/gguf/
```

File: `gpt-oss-20b-Q4_K_M.gguf` (~9 GB)  
Source: `unsloth/gpt-oss-20b-GGUF`

> **Note:** This is the `unsloth` re-quantization, **not** the official competition GGUF
> (`llkh0a/gpt-oss-20b-gguf`).  The competition GGUF is only available inside Kaggle.
> See Section 3 for why this matters.

In [ ]:
GGUF_DIR = Path('/raid/hf_cache/gguf')

models = {
    'gemma-4-26B_q4_0-it.gguf': 'Gemma 4 26B (working ✓)',
    'gpt-oss-20b-Q4_K_M.gguf' : 'GPT-OSS 20B unsloth (non-functional — Bug 8 WNT)',
}

for fname, label in models.items():
    p = GGUF_DIR / fname
    if p.exists():
        size_gb = round(p.stat().st_size / 1e9, 1)
        print(f'✓ {label}')
        print(f'    {p}  ({size_gb} GB)')
    else:
        print(f'✗ {label} — NOT downloaded')
        print(f'    Expected at: {p}')

## 3. GPT-OSS 20B on DGX — Challenges and Investigation (Bug 8)

**Status: Won't Fix on DGX.  Use Kaggle for GPT-OSS scores.**

Getting GPT-OSS 20B to produce valid tool calls on DGX consumed significant investigation
time.  The root cause turned out to be a model identity mismatch — and no workaround exists
without the actual competition GGUF.

---

### Root cause: wrong GGUF

The competition uses `llkh0a/gpt-oss-20b-gguf` (mounted inside Kaggle at a fixed path).
That GGUF is not publicly downloadable.  The closest available model is
`unsloth/gpt-oss-20b-GGUF` — a different quantization of the same base model.

GPT-OSS 20B uses a custom **harmony channel** format for tool calls:

```
<|start|>assistant to=functions.X<|channel|>commentary json<|message|>{args}<|call|>
```

The `<|message|>` token (id=200008) must follow the channel declaration.  The **unsloth**
GGUF assigns near-zero probability to token 200008 in every context tested — it was never
trained on the harmony format that the competition GGUF knows.

---

### All approaches tried (8 total)

| Approach | Config | Result |
|---|---|---|
| **1** | `tool_choice="auto"` (lazy PEG grammar) | `???` degeneration — lazy grammar never fires → PEG validates garbage → HTTP 500 |
| **2** | `tool_choice="required"` (non-lazy PEG) | Correct prefix `to=functions.X<|channel|>analysis` generated, then P(`<|message|>`) ≈ 0 → grammar fallback → gibberish. No `tool_calls` in response. |
| **3** | `tool_choice="none"` | PEG still validates `/v1/chat/completions` output with `--jinja` → HTTP 500 on every call |
| **4** | No `tools=` parameter | PEG still fires (template-generated grammar wins) → HTTP 500 |
| **5** | `grammar: ""` in request body | Ignored — Jinja template grammar overrides client-supplied grammar |
| **6** | `--skip-chat-parsing` + `tool_choice="required"` | Non-lazy grammar fires, produces correct prefix, same `<|message|>` failure |
| **7** | `/completion` endpoint + manual harmony prompt | Gibberish — model can't generate harmony format without `--jinja` developer block |
| **8** | `/tokenize` + BOS token + `/completion` | Bypasses chat parsing but same near-zero P(`<|message|>`) problem — model not harmony-trained |

### Why approach 2 looked like it was working

With `tool_choice="required"`, llama.cpp's non-lazy PEG grammar **forces** the model to
generate the harmony prefix correctly.  The output log shows:

```
to=functions.web_search<|channel|>analysis
```

This looks right.  But the grammar then requires `<|message|>` next.  Log-probabilities
after `<|channel|>analysis` show no trace of token 200008:

```
Top-10 logprobs after '<|channel|>analysis':
  id=36032  tok=' prospect'    p≈0.08
  id=37738  tok='etur'         p≈0.07
  ...
  (token 200008 = <|message|> not in top 10)
```

The constrained sampler falls back to unconstrained sampling and generates English
text instead of a tool call.  The response has no `tool_calls` field.

---

### Current llama-serve.sh state for GPT-OSS

`scripts/llama-serve.sh` still accepts `MODEL=gpt-oss` and serves the unsloth GGUF
with `--jinja --reasoning off`.  `LLMEnv._llm_call_gpt_oss()` uses
`tool_choice="required"` with `max_tokens=32` (fast-fail — was 512, causing ~100s/call
of gibberish generation).  Every call returns `("", [])`.  The code is preserved for
any future attempt with the actual competition GGUF.

**DGX strategy:** Gemma 4 26B for all local harness development.  Kaggle for GPT-OSS scores.

## 4. Starting llama-server (Gemma 4 26B)

Run in a **tmux session** so it survives SSH drops and continues serving during the harness run:

```zsh
tmux new -s llama
cd jed-redteam-attack/
MODEL=gemma bash scripts/llama-serve.sh
# Ctrl+B then D to detach
```

What `llama-serve.sh` does:

1. Checks / builds `jed-llama:latest` Docker image
2. Downloads `gemma-4-26B_q4_0-it.gguf` from HuggingFace if not cached
3. Pauses other long-lived containers to free GPU memory
4. Runs `llama-server` with `--jinja --n-gpu-layers -1 --port 8082`
5. Resumes paused containers on exit

Server is ready when you see:
```
llama server listening at http://0.0.0.0:8082
```

Key flags:

| Flag | Purpose |
|---|---|
| `--jinja` | Apply embedded Jinja2 chat template — injects TypeScript tool schema via `render_tool_namespace` |
| `--n-gpu-layers -1` | Offload all 62 transformer layers to GPU (fits in 128 GB unified memory) |
| `--ctx-size 8192` | 8K context window (matches Kaggle evaluator default) |

In [ ]:
SERVER_PORT = int(os.environ.get('PORT', 8082))
SERVER_BASE = f'http://localhost:{SERVER_PORT}/v1'

def check_server(base_url: str) -> bool:
    try:
        with urllib.request.urlopen(f'{base_url}/models', timeout=5) as r:
            data = json.loads(r.read())
        models = [m['id'] for m in data.get('data', [])]
        print(f'✓ llama-server UP  →  {base_url}')
        print(f'  loaded: {models}')
        return True
    except Exception as e:
        print(f'✗ llama-server not reachable at {base_url}')
        print(f'  {e}')
        print(f'  Start with: tmux new -s llama && MODEL=gemma bash scripts/llama-serve.sh')
        return False

server_ok = check_server(SERVER_BASE)

## 5. Load AttackAlgorithm

In [ ]:
from local_harness import load_attack_class, evaluate_redteam_local, make_env
from aicomp_sdk.attacks import AttackRunConfig

BUDGET_S = int(os.environ.get('BUDGET', 9000))
SEED     = 123
ARTIFACTS = REPO_DIR / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)

AttackAlgorithm = load_attack_class(str(REPO_DIR / 'attack.py'))
print(f'AttackAlgorithm : {AttackAlgorithm}')
print(f'Budget          : {BUDGET_S}s')
print(f'Seed            : {SEED}')
print(f'Artifacts dir   : {ARTIFACTS}')

## 6. Run Against Gemma 4 26B

Ensure llama-server is loaded with `MODEL=gemma` (Cell 4 health check shows `gemma` in model list).

At `BUDGET=9000` this cell runs for ~2.5–3 hours.  Use tmux or run via the CLI instead:

```zsh
tmux new -s harness
MODEL=gemma bash scripts/harness-run.sh
# Ctrl+B D to detach
```

In [ ]:
MODEL_NAME = 'gemma'
os.environ['VLLM_BASE_URL'] = SERVER_BASE
os.environ['VLLM_MODEL']    = MODEL_NAME

assert server_ok, 'Start llama-server first: MODEL=gemma bash scripts/llama-serve.sh'

print(f'=== Gemma 4 26B | budget={BUDGET_S}s | {SERVER_BASE} ===')
config = AttackRunConfig(time_budget_s=BUDGET_S, max_tool_hops=8, seed=SEED)
env    = make_env(use_llm=True, seed=SEED, verbose=False)

t0     = time.time()
result = evaluate_redteam_local(AttackAlgorithm, env, config, verbose=True)
wall_s = round(time.time() - t0, 1)

gemma_summary = {
    'model'                      : MODEL_NAME,
    'score_normalized_0_to_1000' : result.score,
    'score_raw'                  : result.score_raw,
    'findings_count'             : result.findings_count,
    'unique_cells'               : result.unique_cells,
    'evaluation_time_s'          : round(result.time_taken, 1),
    'wall_time_s'                : wall_s,
}
(ARTIFACTS / f'{MODEL_NAME}_summary.json').write_text(json.dumps(gemma_summary, indent=2))
print(json.dumps(gemma_summary, indent=2))

## 7. GPT-OSS 20B Run (Non-Functional on DGX)

The cell below attempts a GPT-OSS run for completeness.  It will succeed in starting
but every LLM call returns `("", [])` — the model never produces valid tool calls
because the unsloth GGUF was not trained on the harmony channel format.  See Section 3.

**For actual GPT-OSS scores, use the Kaggle notebook** (`kaggle_notebook.ipynb`).

In [ ]:
# GPT-OSS run — non-functional on DGX (Bug 8 WNT).
# Uncomment to verify current behavior; expect score=0, findings=0.

# MODEL_NAME = 'gpt_oss'
# os.environ['VLLM_BASE_URL'] = SERVER_BASE
# os.environ['VLLM_MODEL']    = MODEL_NAME
# 
# # Restart llama-server with MODEL=gpt-oss before running:
# # MODEL=gpt-oss bash scripts/llama-serve.sh
# 
# config  = AttackRunConfig(time_budget_s=300, max_tool_hops=8, seed=SEED)  # short budget
# env     = make_env(use_llm=True, seed=SEED, verbose=True)
# result  = evaluate_redteam_local(AttackAlgorithm, env, config, verbose=True)
# print('GPT-OSS result (expected: score=0):', result.score)

print('GPT-OSS run skipped — non-functional on DGX (Bug 8 WNT).')
print('See Section 3 for full investigation details.')
print('Use Kaggle for GPT-OSS scores.')

## 8. Results

In [ ]:
def _load(name):
    p = ARTIFACTS / f'{name}_summary.json'
    return json.loads(p.read_text()) if p.exists() else None

gemma = locals().get('gemma_summary') or _load('gemma')

print('=' * 60)
print('  DGX SPARK GB10 — RESULTS')
print('=' * 60)

if gemma:
    print('\nGemma 4 26B:')
    for k, v in gemma.items():
        print(f'  {k:<36}: {v}')
else:
    print('\nGemma 4 26B: not yet run (run Cell 6)')

print('\nGPT-OSS 20B: Non-functional on DGX (Bug 8 WNT) — see Kaggle submission for score')
print()

## Notes

### DGX vs Kaggle scoring

DGX local scores use the **stub evaluator** (`local_harness.py`), which mirrors the
competition scoring formula but uses our local `aicomp_sdk` fixture set.  Competition
scores come from the Kaggle evaluator (official fixtures, OptimalGuardrail, both models).
DGX and Kaggle scores are **not directly comparable**.

### Gemma vs Kaggle Gemma

The DGX uses `google/gemma-4-26B-A4B-it-qat-q4_0-gguf` (public HuggingFace).  The Kaggle
evaluator uses `llkh0a/gemma-4-26b-a4b-it-ud-q4-k-m-gguf` — a different quantization.
Both use the same base model but different quant methods (Q4_0 vs UD-Q4_K_M).  Results
are similar but not identical.

### Budget

Override budget via environment variable before starting the kernel:

```python
import os; os.environ['BUDGET'] = '300'  # 300s for fast iteration
```

Or use the CLI directly for long runs:

```zsh
BUDGET=9000 MODEL=gemma bash scripts/harness-run.sh
```

### Source

All code: [github.com/msusol/kaggle-ai-agent-security-multi-step-tool-attacks](https://github.com/msusol/kaggle-ai-agent-security-multi-step-tool-attacks)